# C-rate Diagnostics

This notebook evaluates the seven-parameter eSOH diagnostic algorithm on charge data collected at different C-rates. The default configuration runs the BOL C-rate dataset. An EOL dataset is provided in the same data folder; to run it, uncomment the EOL `FILE_PATHS` and `R_by_label` block and comment the BOL block. The saved outputs retain historical column names (`x100`, `y100`, `si_scale_a`, `si_shift_b`), corresponding to the manuscript symbols `x_n,100`, `x_p,100`, `s_V`, and `U_off`.

The C-rate-specific resistance terms in `R_by_label` provide kinetic/IR compensation before the voltage-analysis fit. The resulting summaries are used to compare C-rate robustness and to support the transition-SoC C-rate sensitivity analysis.


In [ ]:
import importlib
import os
import sys
import tempfile
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Return the repository root that contains the shared code and data folders."""
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "code" / "diagnostic_algorithm_lifetime_crate").is_dir() and (candidate / "data" / "bol_eol_crate_data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
CODE_DIR = REPO_ROOT / "code"
PROJECT_DIR = CODE_DIR / "diagnostic_algorithm_lifetime_crate"
CRATE_DATA_DIR = REPO_ROOT / "data" / "bol_eol_crate_data"
OUTPUT_ROOT = PROJECT_DIR / "batch_results" / "A02_crate_diagnostics"
CACHE_DIR = PROJECT_DIR / "_crate_cache"

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "managing_si_burnout_matplotlib"))

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import diagnostic_algorithm_lifetime_crate.config as cfgmod
import diagnostic_algorithm_lifetime_crate.estimation as estmod
import diagnostic_algorithm_lifetime_crate.plot as plotmod
import diagnostic_algorithm_lifetime_crate.user_functions as ufmod

importlib.reload(cfgmod)
importlib.reload(estmod)
importlib.reload(plotmod)
importlib.reload(ufmod)

from diagnostic_algorithm_lifetime_crate.config import VoltageFitConfig
from diagnostic_algorithm_lifetime_crate.run_structured_batch_esoh_parallel import run_structured_batch_esoh

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("C-rate data folder:", CRATE_DATA_DIR)
print("Output folder:", OUTPUT_ROOT)


In [ ]:
# Fitting configuration used for the C-rate diagnostic comparison.
fit_cfg = VoltageFitConfig()

# Enable the two-parameter effective silicon-OCP deformation used in the paper.
fit_cfg.enable_si_drift = True
fit_cfg.de_workers = 1


In [ ]:
# Default: BOL C-rate data for cell 076.
FILE_PATHS = [
    CRATE_DATA_DIR / "GMFEB23S_CELL076_C10C5C3Expansion_1_P25C_15P0PSI_20260206_R0_36_1_2_2818579691.xlsx",
    CRATE_DATA_DIR / "GMFEB23S_CELL076_C20Expansion_1_P25C_15P0PSI_20260124_R0_36_1_2_2818579686.xlsx",
    CRATE_DATA_DIR / "GMFEB23S_CELL076_C100Expansion_1_P25C_15P0PSI_20260124_R0_36_1_2_2818579684.xlsx",
]
R = 0.02
R_by_label = {
    "C/100": 0.0,
    "C/20": 0.0,
    "C/10": R,
    "C/5": R,
    "C/3": R,
}
RUN_NAME = "BOL_crate_diagnostics"

# EOL C-rate data for cell 020. To run the EOL case, comment the BOL block
# above and uncomment this block.
# FILE_PATHS = [
#     CRATE_DATA_DIR / "GMFEB23S_CELL020_Cby100Cby20Cby10Cby5Cby3_20251222_R0_36_2_1_2818579680.xlsx",
# ]
# R_by_label = {
#     "C/100": 0.0,
#     "C/20": 0.0,
#     "C/10": 0.18,
#     "C/5": 0.096,
#     "C/3": 0.0876,
# }
# RUN_NAME = "EOL_crate_diagnostics"

missing = [p for p in FILE_PATHS if not p.exists()]
if missing:
    raise FileNotFoundError("Missing C-rate input files:\n" + "\n".join(str(p) for p in missing))

results_df, out_dir = run_structured_batch_esoh(
    file_paths=FILE_PATHS,
    project_dir=PROJECT_DIR,
    fit_cfg=fit_cfg,
    R_by_label=R_by_label,
    nominal_capacity_ah=2.5,
    run_name=RUN_NAME,
    output_group="A02_crate_diagnostics",
    charge_threshold_a=0.01,
    min_segment_points=300,
    cc_only=True,
    cc_rel_tol=0.05,
    save_plots=True,
    default_R_ohm=0.0,
    parallel=True,
    n_jobs=4,
)

print("out_dir =", out_dir)
print("saved files:")
for p in sorted(Path(out_dir).glob("*")):
    print(" -", p.name)


In [ ]:
# Inspect the constant-current charge segments detected from the selected files.
from diagnostic_algorithm_lifetime_crate.crate_io import structure_segments_from_files

segments = structure_segments_from_files(
    list(FILE_PATHS),
    nominal_capacity_ah=2.5,
    charge_threshold_a=0.01,
    min_segment_points=300,
    cc_only=True,
    cc_rel_tol=0.05,
    cache_dir=CACHE_DIR,
    refresh_cache=False,
)

for i, seg in enumerate(segments):
    print(
        i,
        seg.file,
        seg.segment_id,
        seg.crate_label,
        seg.I_mean_A,
        seg.Ah_throughput,
    )


In [ ]:
# Display the aggregate C-rate diagnostic summary generated by the batch run.
summary_by_crate = pd.read_csv(Path(out_dir) / "crate_parameter_summary_by_crate.csv")
display(summary_by_crate)
